In [1]:
# Cox's Bazar — Service Density Analysis
# Part of the Atlas of Urban Fragility

import osmnx as ox
import geopandas as gpd
import pandas as pd
import folium
import h3
from shapely.geometry import Polygon, Point
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Sanity check
print(f"osmnx:      {ox.__version__}")
print(f"geopandas:  {gpd.__version__}")
print(f"folium:     {folium.__version__}")
print(f"h3:         {h3.__version__}")

# Project paths
PROJECT_ROOT = Path.cwd().parent  # we're in atlas/, so parent is heissj.github.io/
DATA_DIR = PROJECT_ROOT / "data" / "coxs-bazar"
DATA_DIR.mkdir(parents=True, exist_ok=True)
print(f"\nData directory: {DATA_DIR}")

# Verify we're on the right Python
import sys
print(f"\nPython: {sys.executable}")

osmnx:      2.1.0
geopandas:  1.1.3
folium:     0.20.0
h3:         4.4.1

Data directory: /Users/sajjadsharifi/UrbanProjects/heissj.github.io/data/coxs-bazar

Python: /opt/homebrew/Caskroom/miniforge/base/envs/urban/bin/python


In [2]:
# Area of interest: the Rohingya refugee settlements
# (Kutupalong-Balukhali mega-camp area, Ukhia Upazila, Bangladesh)
AREA = {
    'name': "Cox's Bazar Refugee Settlements",
    'bbox': {
        'south': 21.10,
        'north': 21.25,
        'west':  92.12,
        'east':  92.20,
    }
}

center_lat = (AREA['bbox']['south'] + AREA['bbox']['north']) / 2
center_lon = (AREA['bbox']['west']  + AREA['bbox']['east'])  / 2

print(f"Area:   {AREA['name']}")
print(f"Center: {center_lat:.4f}°N, {center_lon:.4f}°E")
print(f"Span:   {AREA['bbox']['north']-AREA['bbox']['south']:.3f}° lat × "
      f"{AREA['bbox']['east']-AREA['bbox']['west']:.3f}° lon")

Area:   Cox's Bazar Refugee Settlements
Center: 21.1750°N, 92.1600°E
Span:   0.150° lat × 0.080° lon


In [3]:
# Quick map to confirm the area looks right
m = folium.Map(location=[center_lat, center_lon], zoom_start=12,
               tiles='OpenStreetMap')

# Draw the bounding box
b = AREA['bbox']
folium.Rectangle(
    bounds=[[b['south'], b['west']], [b['north'], b['east']]],
    color='#cc0000', weight=2,
    fill=True, fill_opacity=0.1,
    tooltip=AREA['name']
).add_to(m)

m

In [4]:
# OSM tags for health facilities — covers hospitals, clinics, pharmacies,
# doctors' offices, and anything tagged as healthcare
tags = {
    'amenity': ['hospital', 'clinic', 'doctors', 'pharmacy'],
    'healthcare': True,
}

# osmnx 2.x bbox format: (west, south, east, north)
bbox_tuple = (b['west'], b['south'], b['east'], b['north'])

print(f"Querying OpenStreetMap for health facilities in {AREA['name']}...")
facilities = ox.features.features_from_bbox(bbox=bbox_tuple, tags=tags)
print(f"\nReturned {len(facilities)} features")
print(f"\nFirst few rows:")
facilities.head()

Querying OpenStreetMap for health facilities in Cox's Bazar Refugee Settlements...

Returned 41 features

First few rows:


geometry   amenity                 name  \
element id                                                                     
node    4939109975  POINT (92.16805 21.21363)  hospital  Kutupalong Hospital   
        5155680645   POINT (92.16095 21.1905)       NaN      MSF Health Care   
        5340024222  POINT (92.17071 21.20466)  hospital                  NaN   
        5340054621  POINT (92.15573 21.17711)  hospital                  NaN   
        5340279422  POINT (92.15381 21.19881)  hospital                  NaN   

                             source source:date healthcare  \
element id                                                   
node    4939109975  MissingMapsKobo        2019        NaN   
        5155680645   GPS, Field map         NaN        yes   
        5340024222              NaN         NaN        NaN   
        5340054621              NaN         NaN        NaN   
        5340279422              NaN         NaN        NaN   

                                                         name:en  \
element id                                                         
node    4939109975                                           NaN   
        5155680645                                           NaN   
        5340024222              Red Cross, Red Crescent Hospital   
        5340054621  Samaritan’s Purse Diptheria Treatment Center   
        5340279422                  MSF Cholera Treatment Centre   

                              operator emergency healthcare:speciality  \
element id                                                               
node    4939109975                 NaN       NaN                   NaN   
        5155680645                 NaN       NaN                   NaN   
        5340024222                 NaN       NaN                   NaN   
        5340054621  Samaritain's Purse       NaN                   NaN   
        5340279422                 NaN       NaN                   NaN   

                   opening_hours phone beds name:my pcode building  
element id                                                          
node    4939109975           NaN   NaN  NaN     NaN   NaN      NaN  
        5155680645           NaN   NaN  NaN     NaN   NaN      NaN  
        5340024222           NaN   NaN  NaN     NaN   NaN      NaN  
        5340054621           NaN   NaN  NaN     NaN   NaN      NaN  
        5340279422           NaN   NaN  NaN     NaN   NaN      NaN

In [5]:
# What kinds of facilities are in the data?
print("Amenity types:")
print(facilities['amenity'].value_counts() if 'amenity' in facilities.columns else "(no amenity column)")

print("\nHealthcare types:")
print(facilities['healthcare'].value_counts() if 'healthcare' in facilities.columns else "(no healthcare column)")

print(f"\nGeometry types:")
print(facilities.geometry.geom_type.value_counts())

Amenity types:
amenity
hospital    20
clinic      19
doctors      1
Name: count, dtype: int64

Healthcare types:
healthcare
hospital    2
yes         1
Name: count, dtype: int64

Geometry types:
Point      31
Polygon    10
Name: count, dtype: int64


In [6]:
# Convert polygon geometries to their centroids; points stay as points
# Project to Web Mercator first for accurate centroid math, then back to WGS84
facilities_proj = facilities.to_crs(epsg=3857)
facilities_proj['geometry'] = facilities_proj.geometry.centroid
facilities_pts = facilities_proj.to_crs(epsg=4326)

print(f"{len(facilities_pts)} facilities, all as points now")
print(f"Geometry types: {facilities_pts.geometry.geom_type.value_counts().to_dict()}")

41 facilities, all as points now
Geometry types: {'Point': 41}


In [7]:
# H3 resolution 8 → hexagons roughly 0.5 km across
# (Good resolution for a camp-scale analysis; res 7 = too coarse, res 9 = too fine)
H3_RESOLUTION = 8

# Define our area as an H3 polygon (vertices in lat, lng order)
vertices = [
    (b['south'], b['west']),
    (b['north'], b['west']),
    (b['north'], b['east']),
    (b['south'], b['east']),
]
h3_poly = h3.LatLngPoly(vertices)

# Get all H3 cells inside this polygon
hex_ids = h3.h3shape_to_cells(h3_poly, H3_RESOLUTION)
print(f"Generated {len(hex_ids)} H3 hexes at resolution {H3_RESOLUTION}")
print(f"First few hex IDs: {hex_ids[:3]}")

Generated 169 H3 hexes at resolution 8
First few hex IDs: ['883cd12181fffff', '883cd12abbfffff', '883cd121a3fffff']


In [8]:
# Build a GeoDataFrame where each row is one hexagon
hex_records = []
for h in hex_ids:
    # h3.cell_to_boundary returns list of (lat, lng) tuples
    # Shapely expects (lng, lat) order
    boundary = h3.cell_to_boundary(h)
    poly = Polygon([(lng, lat) for lat, lng in boundary])
    hex_records.append({'h3_id': h, 'geometry': poly})

hexes = gpd.GeoDataFrame(hex_records, crs='EPSG:4326')

# Spatial join: which hex contains each facility?
joined = gpd.sjoin(facilities_pts, hexes, how='left', predicate='within')

# Count facilities per hex
counts = joined.groupby('h3_id').size().rename('facility_count')
hexes = hexes.merge(counts, on='h3_id', how='left')
hexes['facility_count'] = hexes['facility_count'].fillna(0).astype(int)

# Quick stats
print(f"Total hexes:                {len(hexes)}")
print(f"Hexes with ≥1 facility:     {(hexes['facility_count'] > 0).sum()}")
print(f"Hexes with ≥3 facilities:   {(hexes['facility_count'] >= 3).sum()}")
print(f"Max facilities in one hex:  {hexes['facility_count'].max()}")
print(f"\nDistribution of facility counts:")
print(hexes['facility_count'].value_counts().sort_index())

Total hexes:                169
Hexes with ≥1 facility:     22
Hexes with ≥3 facilities:   2
Max facilities in one hex:  4

Distribution of facility counts:
facility_count
0    147
1      6
2     14
3      1
4      1
Name: count, dtype: int64


In [10]:
# Set up the map
m = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=13,
    tiles='OpenStreetMap'
)

# Style function: color hexes by facility count
def style_fn(feature):
    count = feature['properties']['facility_count']
    if count == 0:
        return {'fillColor': '#ffffff', 'color': '#cccccc',
                'weight': 0.3, 'fillOpacity': 0.0}
    elif count == 1:
        return {'fillColor': '#9ecae1', 'color': '#4292c6',
                'weight': 0.6, 'fillOpacity': 0.6}
    elif count == 2:
        return {'fillColor': '#4292c6', 'color': '#2171b5',
                'weight': 0.6, 'fillOpacity': 0.7}
    else:  # 3+
        return {'fillColor': '#08519c', 'color': '#08306b',
                'weight': 0.8, 'fillOpacity': 0.85}

# Add the hexes
folium.GeoJson(
    hexes.to_json(),
    name='Health facility density (per hex)',
    style_function=style_fn,
    tooltip=folium.GeoJsonTooltip(
        fields=['facility_count'],
        aliases=['Health facilities:']
    )
).add_to(m)

# Add individual facility points on top for context
for _, row in facilities_pts.iterrows():
    name = (row.get('name:en') or row.get('name')
            or str(row.get('amenity', 'Health facility')))
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=3, color='#222', weight=1,
        fill=True, fillColor='white', fillOpacity=1.0,
        tooltip=name
    ).add_to(m)

# Add a simple legend (fixed-position HTML overlay)
legend_html = '''
<div style="position: fixed; bottom: 30px; left: 30px;
            background-color: white; border: 1px solid #999;
            padding: 10px 14px; font-family: Arial, sans-serif;
            font-size: 12px; z-index: 9999;
            box-shadow: 0 2px 6px rgba(0,0,0,0.15); line-height: 1.6;">
    <div style="font-weight: bold; margin-bottom: 6px;">
        Health facilities per hex
    </div>
    <div><span style="background:#08519c;opacity:0.85;width:18px;height:14px;
                      display:inline-block;vertical-align:middle;margin-right:6px;"></span>3 or more</div>
    <div><span style="background:#4292c6;opacity:0.7;width:18px;height:14px;
                      display:inline-block;vertical-align:middle;margin-right:6px;"></span>2</div>
    <div><span style="background:#9ecae1;opacity:0.6;width:18px;height:14px;
                      display:inline-block;vertical-align:middle;margin-right:6px;"></span>1</div>
    <div><span style="background:white;border:1px solid #ccc;width:18px;height:14px;
                      display:inline-block;vertical-align:middle;margin-right:6px;"></span>0</div>
    <div style="border-top:1px solid #eee;margin-top:6px;padding-top:4px;font-size:11px;color:#666;">
        Hex resolution: H3 res 8 (~0.5 km)
    </div>
</div>
'''
m.get_root().html.add_child(folium.Element(legend_html))

# Display
m

In [11]:
output_path = PROJECT_ROOT / "atlas" / "coxs-bazar-map.html"
m.save(str(output_path))
print(f"Map saved to: {output_path}")

Map saved to: /Users/sajjadsharifi/UrbanProjects/heissj.github.io/atlas/coxs-bazar-map.html
